# Layer 1: Message Rate Characterisation


In [ ]:
import os
import numpy as np
import pandas as pd

from mda.loader import load_trades
from mda.timestamps import add_ts_columns
from mda import EXCHANGES, DATA_DIR
from mda.layer1.rates import (
    layer1_summary,
    plot_rate_timeseries,
    plot_intermessage_histogram,
    plot_burst_scatter,
    plot_session_profile,
    plot_volume_heatmap,
)
from mda.plots.charts import save_figure


In [ ]:
DATA_DIR = "/data/parquet"
REPORTS_DIR = "/data/reports"
os.makedirs(REPORTS_DIR, exist_ok=True)


In [ ]:
df = load_trades(DATA_DIR, exchanges=EXCHANGES, add_ts_cols=True)
print("Shape:", df.shape)
print("\nTrades per exchange:")
print(df["exchange"].value_counts())


In [ ]:
from mda.layer1.rates import layer1_summary

results = layer1_summary(df)
print("Rate percentiles per exchange:")
print(results["rate_pcts"])


In [ ]:
# Plot R1: Rate timeseries
from mda.plots.charts import plot_rate_timeseries, save_figure

fig = plot_rate_timeseries(results["rate_ts"], results["rate_pcts"])
fig.show()
save_figure(fig, "R1_rate_timeseries", REPORTS_DIR)


In [ ]:
# Plot R2: Inter-message interval histogram
from mda.layer1.rates import plot_intermessage_histogram

fig = plot_intermessage_histogram(df)
fig.show()
save_figure(fig, "R2_intermessage_histogram", REPORTS_DIR)


In [ ]:
# Plot R3: Burst scatter
from mda.layer1.rates import plot_burst_scatter

fig = plot_burst_scatter(results["burst_events"])
fig.show()
save_figure(fig, "R3_burst_scatter", REPORTS_DIR)


In [ ]:
# Plot R4: Session profile (intraday message rate by hour)
from mda.layer1.rates import plot_session_profile

fig = plot_session_profile(df)
fig.show()
save_figure(fig, "R4_session_profile", REPORTS_DIR)


In [ ]:
# Plot R5: Volume heatmap per exchange
from mda.layer1.rates import plot_volume_heatmap

for exch in EXCHANGES:
    df_ex = df[df["exchange"] == exch]
    if df_ex.empty:
        print(f"No data for {exch}, skipping.")
        continue
    fig = plot_volume_heatmap(df_ex, exchange=exch)
    fig.show()
    save_figure(fig, f"R5_volume_heatmap_{exch}", REPORTS_DIR)


In [ ]:
# Summary statistics
print("=== Layer 1 Summary ===")
print(f"Total messages : {len(df):,}")
print(f"Exchanges      : {df['exchange'].nunique()}")
print(f"Date range     : {df['ts_event'].min()} -> {df['ts_event'].max()}")
print("\nPeak rate (msgs/s) by exchange:")
print(results["rate_pcts"].loc["p99"] if "p99" in results["rate_pcts"].index else results["rate_pcts"].tail(1))
